# 1. Carga de Datos - MLOps Pipeline

Este notebook implementa la carga inicial de datos del proyecto.

**Objetivo**: Cargar el dataset `Churn_Modelling.csv` y realizar validaciones básicas.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

# Configurar el path al directorio raíz del proyecto
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

print(f"Directorio del proyecto: {project_root}")
print(f"Python version: {sys.version}")

Directorio del proyecto: c:\Users\Asus\Desktop\Proyecto
Python version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]


## 1.1 Función Genérica de Carga de Datos

In [2]:
def cargar_datos(ruta_archivo, encoding='utf-8', delimiter=',', **kwargs):
    """
    Función genérica para cargar archivos CSV.
    
    Parámetros:
    -----------
    ruta_archivo : str o Path
        Ruta al archivo CSV a cargar
    encoding : str, default='utf-8'
        Codificación del archivo
    delimiter : str, default=','
        Delimitador del CSV
    **kwargs : dict
        Argumentos adicionales para pandas.read_csv
    
    Retorna:
    --------
    pd.DataFrame : DataFrame con los datos cargados
    dict : Metadata sobre la carga (filas, columnas, tamaño del archivo)
    """
    try:
        # Validar que el archivo existe
        if not os.path.exists(ruta_archivo):
            raise FileNotFoundError(f"El archivo {ruta_archivo} no existe")
        
        # Obtener información del archivo
        tamaño_archivo = os.path.getsize(ruta_archivo) / (1024 * 1024)  # MB
        
        print(f"Cargando archivo: {ruta_archivo}")
        print(f"Tamaño del archivo: {tamaño_archivo:.2f} MB")
        
        # Cargar el dataset
        df = pd.read_csv(ruta_archivo, encoding=encoding, delimiter=delimiter, **kwargs)
        
        # Metadata
        metadata = {
            'filas': df.shape[0],
            'columnas': df.shape[1],
            'tamaño_mb': tamaño_archivo,
            'columnas_nombres': df.columns.tolist(),
            'memoria_mb': df.memory_usage(deep=True).sum() / (1024 * 1024)
        }
        
        print(f"\n✓ Datos cargados exitosamente")
        print(f"  - Filas: {metadata['filas']:,}")
        print(f"  - Columnas: {metadata['columnas']}")
        print(f"  - Memoria utilizada: {metadata['memoria_mb']:.2f} MB")
        
        return df, metadata
        
    except Exception as e:
        print(f"✗ Error al cargar datos: {str(e)}")
        raise

## 1.2 Cargar Dataset del Proyecto

In [3]:
# Definir la ruta al archivo de datos
ruta_datos = project_root / 'Churn_Modelling.csv'

# Cargar los datos
df, metadata = cargar_datos(ruta_datos)

# Mostrar las primeras filas
print("\nPrimeras 5 filas del dataset:")
df.head()

Cargando archivo: c:\Users\Asus\Desktop\Proyecto\Churn_Modelling.csv
Tamaño del archivo: 0.65 MB

✓ Datos cargados exitosamente
  - Filas: 10,000
  - Columnas: 14
  - Memoria utilizada: 2.41 MB

Primeras 5 filas del dataset:


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 1.3 Validación Inicial de Datos

In [4]:
def validacion_inicial(df):
    """
    Realiza validaciones básicas sobre el DataFrame cargado.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a validar
    
    Retorna:
    --------
    dict : Diccionario con resultados de la validación
    """
    validacion = {}
    
    # 1. Verificar si hay filas duplicadas
    duplicados = df.duplicated().sum()
    validacion['duplicados'] = {
        'total': duplicados,
        'porcentaje': (duplicados / len(df)) * 100
    }
    
    # 2. Verificar valores nulos
    nulos = df.isnull().sum()
    validacion['nulos'] = {
        'por_columna': nulos[nulos > 0].to_dict(),
        'total': nulos.sum(),
        'porcentaje_total': (nulos.sum() / (len(df) * len(df.columns))) * 100
    }
    
    # 3. Tipos de datos
    validacion['tipos_datos'] = df.dtypes.astype(str).to_dict()
    
    # 4. Distribución de tipos
    validacion['distribucion_tipos'] = df.dtypes.value_counts().to_dict()
    
    # Imprimir resumen
    print("\n" + "="*60)
    print("VALIDACIÓN INICIAL DE DATOS")
    print("="*60)
    
    print(f"\n1. Filas duplicadas: {duplicados} ({validacion['duplicados']['porcentaje']:.2f}%)")
    
    print(f"\n2. Valores nulos totales: {validacion['nulos']['total']} ({validacion['nulos']['porcentaje_total']:.2f}%)")
    if validacion['nulos']['por_columna']:
        print("   Columnas con nulos:")
        for col, count in validacion['nulos']['por_columna'].items():
            print(f"   - {col}: {count} ({(count/len(df))*100:.2f}%)")
    else:
        print("   ✓ No hay valores nulos")
    
    print(f"\n3. Distribución de tipos de datos:")
    for tipo, count in validacion['distribucion_tipos'].items():
        print(f"   - {tipo}: {count} columnas")
    
    return validacion

# Ejecutar validación
validacion = validacion_inicial(df)


VALIDACIÓN INICIAL DE DATOS

1. Filas duplicadas: 0 (0.00%)

2. Valores nulos totales: 0 (0.00%)
   ✓ No hay valores nulos

3. Distribución de tipos de datos:
   - int64: 9 columnas
   - object: 3 columnas
   - float64: 2 columnas


## 1.4 Información General del Dataset

In [5]:
# Información general
print("\nInformación del DataFrame:")
df.info()

print("\nEstadísticas descriptivas:")
df.describe(include='all').T


Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB

Estadísticas descriptivas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,NaN,NaN,NaN,5000.5,2886.89568,1.0,2500.75,5000.5,7500.25,10000.0
CustomerId,10000.0,NaN,NaN,NaN,15690940.5694,71936.186123,15565701.0,15628528.25,15690738.0,15753233.75,15815690.0
Surname,10000,2932,Smith,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CreditScore,10000.0,NaN,NaN,NaN,650.5288,96.653299,350.0,584.0,652.0,718.0,850.0
Geography,10000,3,France,5014,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,10000,2,Male,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,10000.0,NaN,NaN,NaN,38.9218,10.487806,18.0,32.0,37.0,44.0,92.0
Tenure,10000.0,NaN,NaN,NaN,5.0128,2.892174,0.0,3.0,5.0,7.0,10.0
Balance,10000.0,NaN,NaN,NaN,76485.889288,62397.405202,0.0,0.0,97198.54,127644.24,250898.09
NumOfProducts,10000.0,NaN,NaN,NaN,1.5302,0.581654,1.0,1.0,1.0,2.0,4.0


## 1.5 Guardar Datos Procesados (Opcional)

In [6]:
# Crear directorio de datos procesados si no existe
datos_procesados_dir = project_root / 'mlops_pipeline' / 'data' / 'raw'
datos_procesados_dir.mkdir(parents=True, exist_ok=True)

# Guardar una copia del dataset cargado
ruta_salida = datos_procesados_dir / 'churn_raw.csv'
df.to_csv(ruta_salida, index=False)
print(f"\n✓ Dataset guardado en: {ruta_salida}")


✓ Dataset guardado en: c:\Users\Asus\Desktop\Proyecto\mlops_pipeline\data\raw\churn_raw.csv


## 1.6 Exportar Variables para Siguientes Notebooks

In [7]:
# Guardar el DataFrame en formato pickle para uso en siguientes notebooks
import pickle

output_dir = project_root / 'mlops_pipeline' / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'df_raw.pkl', 'wb') as f:
    pickle.dump(df, f)

print(f"\n✓ DataFrame exportado para siguientes etapas")
print(f"\nResumen final:")
print(f"  - Total de registros: {len(df):,}")
print(f"  - Total de características: {len(df.columns)}")
print(f"  - Datos listos para EDA")


✓ DataFrame exportado para siguientes etapas

Resumen final:
  - Total de registros: 10,000
  - Total de características: 14
  - Datos listos para EDA
